# Praktikum Bab 11: CUDA di Google Colab

Buku *Sistem Terdistribusi dan Komputasi Berkinerja Tinggi*, Subbab 11.9 (Kode 11.13).

1. Pilih **Runtime -> Change runtime type -> T4 GPU**.
2. Jalankan sel dari atas ke bawah. Baris `%%writefile` harus paling atas di selnya.
3. Bila GPU yang didapat bukan T4, ganti `sm_75` sesuai compute capability di keluaran `nvidia-smi`.

In [ ]:
!nvidia-smi                      # GPU apa yang didapat, versi driver
!nvcc --version                  # versi CUDA Toolkit

In [ ]:
# compute capability GPU ini, untuk opsi -arch
!nvidia-smi --query-gpu=name,compute_cap --format=csv

## Penjumlahan vektor (Kode 11.7)

In [ ]:
%%writefile vadd.cu
// vadd.cu -- program penjumlahan vektor lengkap (Kode 11.2, 11.6, 11.7).
// Lima langkah: cudaMalloc, salin masuk, kernel, salin keluar, cudaFree.
//
// Kompilasi: nvcc -O2 -arch=sm_75 vadd.cu -o vadd    (sm_75 = T4; lihat Makefile)
// Jalankan : ./vadd
#include <cstdio>
#include <cstdlib>
// Hentikan program bila panggilan CUDA gagal, lengkap dengan lokasinya (Kode 11.6).
// Makro ditulis di setiap berkas agar tiap berkas dapat disalin sendiri ke Colab.
#define CEK(x) do { cudaError_t e = (x); if (e != cudaSuccess) {             \
        fprintf(stderr, "CUDA error: %s di %s:%d\n", cudaGetErrorString(e),  \
                __FILE__, __LINE__);                                          \
        exit(1); } } while (0)

// __global__: dipanggil dari host, dijalankan di device oleh banyak thread
__global__ void tambah(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // indeks global thread ini
    if (i < n)                                        // penjaga batas
        c[i] = a[i] + b[i];
}

int main(void) {
    int n = 1 << 24;                                        // 16 juta elemen
    size_t bytes = n * sizeof(float);
    float *a = (float *)malloc(bytes), *b = (float *)malloc(bytes),
          *c = (float *)malloc(bytes);
    if (!a || !b || !c) { fprintf(stderr, "malloc gagal\n"); return 1; }
    for (int i = 0; i < n; i++) { a[i] = 1.0f; b[i] = 2.0f; }

    float *da, *db, *dc;                                    // 1. alokasi
    CEK(cudaMalloc(&da, bytes)); CEK(cudaMalloc(&db, bytes)); CEK(cudaMalloc(&dc, bytes));
    CEK(cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice));  // 2. salin masuk
    CEK(cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice));

    int blok = 256, grid = (n + blok - 1) / blok;           // 3. kernel
    tambah<<<grid, blok>>>(da, db, dc, n);
    CEK(cudaGetLastError());                                // error konfigurasi peluncuran
    CEK(cudaDeviceSynchronize());                           // error saat kernel berjalan

    CEK(cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost));  // 4. salin keluar
    printf("c[0] = %.1f, c[n-1] = %.1f\n", c[0], c[n - 1]); // harus 3.0

    // periksa SEMUA elemen, bukan hanya dua
    int salah = 0;
    for (int i = 0; i < n; i++) salah += (c[i] != 3.0f);
    printf("%s: %d elemen salah dari %d\n", salah ? "GAGAL" : "BENAR", salah, n);

    CEK(cudaFree(da)); CEK(cudaFree(db)); CEK(cudaFree(dc)); // 5. bebaskan
    free(a); free(b); free(c);
    return salah ? 1 : 0;
}


In [ ]:
!nvcc -O2 -arch=sm_75 vadd.cu -o vadd && ./vadd

## Mengukur waktu kernel dan transfer (Kode 11.12)

Keluaran CSV dipakai untuk mengisi Tabel 11.7.

In [ ]:
%%writefile vadd_ukur.cu
// vadd_ukur.cu -- mengukur penjumlahan vektor dengan cudaEvent (Kode 11.12):
// waktu CPU, salin masuk, kernel, salin keluar, dan total, untuk mengisi
// tabel praktikum (Tabel 11.7). Waktu kernel = median beberapa ulangan
// setelah pemanasan.
//
// Kompilasi: nvcc -O2 -arch=sm_75 vadd_ukur.cu -o vadd_ukur
// Jalankan : ./vadd_ukur 24        (n = 2^24; default 24)
//            ./vadd_ukur 26 pinned (memakai pinned memory, cudaMallocHost)
// Keluaran : baris CSV  n,cpu_ms,h2d_ms,kernel_ms,d2h_ms,gpu_total_ms,kernel_GBs,speedup_kernel,speedup_total
#include <algorithm>
#include <chrono>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <vector>
// Hentikan program bila panggilan CUDA gagal, lengkap dengan lokasinya (Kode 11.6).
// Makro ditulis di setiap berkas agar tiap berkas dapat disalin sendiri ke Colab.
#define CEK(x) do { cudaError_t e = (x); if (e != cudaSuccess) {             \
        fprintf(stderr, "CUDA error: %s di %s:%d\n", cudaGetErrorString(e),  \
                __FILE__, __LINE__);                                          \
        exit(1); } } while (0)

__global__ void tambah(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        c[i] = a[i] + b[i];
}

// versi CPU satu thread sebagai pembanding dan pemeriksa kebenaran
static void tambah_cpu(const float *a, const float *b, float *c, int n) {
    for (int i = 0; i < n; i++) c[i] = a[i] + b[i];
}

int main(int argc, char **argv) {
    int pangkat = argc > 1 ? atoi(argv[1]) : 24;
    bool pinned = argc > 2 && strcmp(argv[2], "pinned") == 0;
    if (pangkat < 10 || pangkat > 29) { fprintf(stderr, "pangkat 10..29\n"); return 1; }
    int n = 1 << pangkat;
    size_t bytes = (size_t)n * sizeof(float);

    float *a, *b, *c, *ref;
    if (pinned) {                       // memori host terkunci: DMA langsung
        CEK(cudaMallocHost(&a, bytes)); CEK(cudaMallocHost(&b, bytes)); CEK(cudaMallocHost(&c, bytes));
    } else {
        a = (float *)malloc(bytes); b = (float *)malloc(bytes); c = (float *)malloc(bytes);
    }
    ref = (float *)malloc(bytes);
    if (!a || !b || !c || !ref) { fprintf(stderr, "alokasi host gagal\n"); return 1; }
    for (int i = 0; i < n; i++) { a[i] = (float)(i % 1000); b[i] = 0.5f * (i % 7); }
    memset(c, 0, bytes);                // sentuh halaman c lebih dulu agar salin keluar
    memset(ref, 0, bytes);              // tidak ikut menanggung biaya page fault pertama

    // CPU: median dari 3 ulangan
    std::vector<double> tcpu;
    for (int r = 0; r < 3; r++) {
        auto t0 = std::chrono::steady_clock::now();
        tambah_cpu(a, b, ref, n);
        auto t1 = std::chrono::steady_clock::now();
        tcpu.push_back(std::chrono::duration<double, std::milli>(t1 - t0).count());
    }
    std::sort(tcpu.begin(), tcpu.end());
    double cpu_ms = tcpu[1];

    float *da, *db, *dc;
    CEK(cudaMalloc(&da, bytes)); CEK(cudaMalloc(&db, bytes)); CEK(cudaMalloc(&dc, bytes));
    int blok = 256, grid = (n + blok - 1) / blok;

    cudaEvent_t mulai, selesai;
    CEK(cudaEventCreate(&mulai));
    CEK(cudaEventCreate(&selesai));
    float ms = 0.0f;

    // pemanasan: konteks CUDA, kode kernel, alokasi pertama (tidak diukur)
    tambah<<<grid, blok>>>(da, db, dc, n);
    CEK(cudaGetLastError());
    CEK(cudaDeviceSynchronize());

    // salin masuk (host ke device)
    CEK(cudaEventRecord(mulai));
    CEK(cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice));
    CEK(cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice));
    CEK(cudaEventRecord(selesai));
    CEK(cudaEventSynchronize(selesai));
    CEK(cudaEventElapsedTime(&ms, mulai, selesai));
    float h2d_ms = ms;

    // kernel: median dari 11 ulangan
    std::vector<float> tk;
    for (int r = 0; r < 11; r++) {
        CEK(cudaEventRecord(mulai));
        tambah<<<grid, blok>>>(da, db, dc, n);
        CEK(cudaEventRecord(selesai));
        CEK(cudaEventSynchronize(selesai));               // tunggu event selesai
        CEK(cudaGetLastError());
        CEK(cudaEventElapsedTime(&ms, mulai, selesai));
        tk.push_back(ms);
    }
    std::sort(tk.begin(), tk.end());
    float kernel_ms = tk[tk.size() / 2];

    // salin keluar (device ke host)
    CEK(cudaEventRecord(mulai));
    CEK(cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost));
    CEK(cudaEventRecord(selesai));
    CEK(cudaEventSynchronize(selesai));
    CEK(cudaEventElapsedTime(&ms, mulai, selesai));
    float d2h_ms = ms;

    int salah = 0;
    for (int i = 0; i < n; i++) salah += (c[i] != ref[i]);
    float total_ms = h2d_ms + kernel_ms + d2h_ms;
    double gbs = 3.0 * bytes / kernel_ms / 1e6;          // baca a, b, tulis c

    fprintf(stderr, "n = 2^%d (%s), %s; kernel: %.3f ms, bandwidth efektif: %.1f GB/s\n",
            pangkat, pinned ? "pinned" : "pageable", salah ? "HASIL SALAH" : "hasil benar",
            kernel_ms, gbs);
    printf("n,cpu_ms,h2d_ms,kernel_ms,d2h_ms,gpu_total_ms,kernel_GBs,speedup_kernel,speedup_total\n");
    printf("%d,%.3f,%.3f,%.3f,%.3f,%.3f,%.1f,%.2f,%.2f\n", n, cpu_ms, h2d_ms, kernel_ms, d2h_ms,
           total_ms, gbs, cpu_ms / kernel_ms, cpu_ms / total_ms);

    CEK(cudaEventDestroy(mulai)); CEK(cudaEventDestroy(selesai));
    CEK(cudaFree(da)); CEK(cudaFree(db)); CEK(cudaFree(dc));
    if (pinned) { CEK(cudaFreeHost(a)); CEK(cudaFreeHost(b)); CEK(cudaFreeHost(c)); }
    else { free(a); free(b); free(c); }
    free(ref);
    return salah ? 1 : 0;
}


In [ ]:
!nvcc -O2 -arch=sm_75 vadd_ukur.cu -o vadd_ukur
!for p in 20 22 24 26; do ./vadd_ukur $p 2>/dev/null | tail -1; done
!./vadd_ukur 26 pinned        # bandingkan dengan pinned memory

## Stencil 1D dengan shared memory (Kode 11.5)

In [ ]:
%%writefile stencil1d.cu
// stencil1d.cu -- stencil 1D radius R = 3 dengan shared memory dan
// __syncthreads (Kode 11.5), dibandingkan versi tanpa shared memory dan versi CPU.
//
// Kompilasi: nvcc -O2 -arch=sm_75 stencil1d.cu -o stencil1d
// Jalankan : ./stencil1d          (n = 2^24 + 5, sengaja bukan kelipatan BLOK)
//            ./stencil1d 1000
#include <cmath>
#include <cstdio>
#include <cstdlib>

// Hentikan program bila panggilan CUDA gagal, lengkap dengan lokasinya (Kode 11.6).
// Makro ditulis di setiap berkas agar tiap berkas dapat disalin sendiri ke Colab.
#define CEK(x) do { cudaError_t e = (x); if (e != cudaSuccess) {             \
        fprintf(stderr, "CUDA error: %s di %s:%d\n", cudaGetErrorString(e),  \
                __FILE__, __LINE__);                                          \
        exit(1); } } while (0)

#define R 3
#define BLOK 256

__global__ void stencil1d(const float *in, float *out, int n) {
    __shared__ float tmp[BLOK + 2 * R];          // dibagi semua thread satu block
    int g = blockIdx.x * blockDim.x + threadIdx.x;  // indeks global
    int l = threadIdx.x + R;                         // indeks lokal di tmp
    tmp[l] = (g < n) ? in[g] : 0.0f;
    if (threadIdx.x < R) {                           // R thread pertama memuat halo
        tmp[l - R]    = (g - R >= 0)   ? in[g - R]    : 0.0f;
        tmp[l + BLOK] = (g + BLOK < n) ? in[g + BLOK] : 0.0f;
    }
    __syncthreads();                    // tunggu sampai seluruh tmp terisi
    if (g < n) {
        float s = 0.0f;
        for (int o = -R; o <= R; o++) s += tmp[l + o];
        out[g] = s;
    }
}

// Versi tanpa shared memory: setiap masukan dibaca 2R + 1 = 7 kali dari memori
// global oleh thread yang berbeda (cache L1/L2 membantu sebagian).
__global__ void stencil1d_global(const float *in, float *out, int n) {
    int g = blockIdx.x * blockDim.x + threadIdx.x;
    if (g < n) {
        float s = 0.0f;
        for (int o = -R; o <= R; o++) {
            int j = g + o;
            s += (j >= 0 && j < n) ? in[j] : 0.0f;
        }
        out[g] = s;
    }
}

// pembanding CPU dengan batas nol yang sama
static void stencil_cpu(const float *in, float *out, int n) {
    for (int g = 0; g < n; g++) {
        float s = 0.0f;
        for (int o = -R; o <= R; o++) {
            int j = g + o;
            if (j >= 0 && j < n) s += in[j];
        }
        out[g] = s;
    }
}

static float ukur(void (*k)(const float *, float *, int), const float *din, float *dout, int n) {
    int grid = (n + BLOK - 1) / BLOK;
    cudaEvent_t a, b;
    CEK(cudaEventCreate(&a)); CEK(cudaEventCreate(&b));
    k<<<grid, BLOK>>>(din, dout, n);              // pemanasan
    CEK(cudaGetLastError());
    CEK(cudaEventRecord(a));
    for (int r = 0; r < 10; r++) k<<<grid, BLOK>>>(din, dout, n);
    CEK(cudaEventRecord(b));
    CEK(cudaEventSynchronize(b));
    float ms; CEK(cudaEventElapsedTime(&ms, a, b));
    CEK(cudaEventDestroy(a)); CEK(cudaEventDestroy(b));
    return ms / 10;
}

static int periksa(const float *x, const float *ref, int n) {
    int salah = 0;
    for (int i = 0; i < n; i++)
        if (fabsf(x[i] - ref[i]) > 1e-3f * fmaxf(1.0f, fabsf(ref[i]))) salah++;
    return salah;
}

int main(int argc, char **argv) {
    int n = argc > 1 ? atoi(argv[1]) : (1 << 24) + 5;
    size_t bytes = (size_t)n * sizeof(float);
    float *in = (float *)malloc(bytes), *out = (float *)malloc(bytes), *ref = (float *)malloc(bytes);
    for (int i = 0; i < n; i++) in[i] = (float)((i * 37) % 101) / 10.0f;
    stencil_cpu(in, ref, n);

    float *din, *dout;
    CEK(cudaMalloc(&din, bytes)); CEK(cudaMalloc(&dout, bytes));
    CEK(cudaMemcpy(din, in, bytes, cudaMemcpyHostToDevice));

    // blockDim.x HARUS sama dengan BLOK, karena kernel memuat halo kanan di tmp[l + BLOK]
    float ms_sh = ukur(stencil1d, din, dout, n);
    CEK(cudaMemcpy(out, dout, bytes, cudaMemcpyDeviceToHost));
    int salah_sh = periksa(out, ref, n);

    CEK(cudaMemset(dout, 0, bytes));
    float ms_gl = ukur(stencil1d_global, din, dout, n);
    CEK(cudaMemcpy(out, dout, bytes, cudaMemcpyDeviceToHost));
    int salah_gl = periksa(out, ref, n);

    printf("n = %d, R = %d, BLOK = %d\n", n, R, BLOK);
    printf("shared memory : %.3f ms, %s (%d salah)\n", ms_sh, salah_sh ? "GAGAL" : "benar", salah_sh);
    printf("memori global : %.3f ms, %s (%d salah)\n", ms_gl, salah_gl ? "GAGAL" : "benar", salah_gl);
    printf("contoh: out[0] = %.2f, out[n/2] = %.2f, out[n-1] = %.2f\n", ref[0], ref[n / 2], ref[n - 1]);

    CEK(cudaFree(din)); CEK(cudaFree(dout));
    free(in); free(out); free(ref);
    return (salah_sh || salah_gl) ? 1 : 0;
}


In [ ]:
!nvcc -O2 -arch=sm_75 stencil1d.cu -o stencil1d && ./stencil1d

## Perkalian matriks naif (Kode 11.11)

Ukur untuk beberapa n, lalu ubah ukuran block (8, 16, 32). Block 64x64 sengaja gagal.

In [ ]:
%%writefile matmul.cu
// matmul.cu -- perkalian matriks naif: satu thread per elemen C (Kode 11.11).
// Mengukur waktu kernel dengan cudaEvent, menghitung GFLOPS = 2 n^3 / t,
// memeriksa hasil terhadap CPU, dan menerima ukuran block sebagai argumen
// untuk percobaan 8x8, 16x16, 32x32 (dan 64x64 yang gagal diluncurkan).
//
// Kompilasi: nvcc -O2 -arch=sm_75 matmul.cu -o matmul
// Jalankan : ./matmul 1024          (n = 1024, block 16x16)
//            ./matmul 1024 32       (block 32x32)
//            ./matmul 512 64        (block 64x64 = 4096 thread: gagal, dan terdeteksi)
//            ./matmul 2048 16 cpu   (paksa pembanding CPU penuh; lambat untuk n besar)
// Keluaran : baris CSV  n,blok,cpu_ms,h2d_ms,kernel_ms,d2h_ms,gpu_total_ms,GFLOPS,galat_relatif
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <vector>

// Hentikan program bila panggilan CUDA gagal, lengkap dengan lokasinya (Kode 11.6).
// Makro ditulis di setiap berkas agar tiap berkas dapat disalin sendiri ke Colab.
#define CEK(x) do { cudaError_t e = (x); if (e != cudaSuccess) {             \
        fprintf(stderr, "CUDA error: %s di %s:%d\n", cudaGetErrorString(e),  \
                __FILE__, __LINE__);                                          \
        exit(1); } } while (0)

// C = A x B, semua matriks n x n, float, row-major
__global__ void matmul(const float *A, const float *B, float *C, int n) {
    int baris = blockIdx.y * blockDim.y + threadIdx.y;
    int kolom = blockIdx.x * blockDim.x + threadIdx.x;
    if (baris < n && kolom < n) {
        float jumlah = 0.0f;
        for (int k = 0; k < n; k++)
            jumlah += A[baris * n + k] * B[k * n + kolom];
        C[baris * n + kolom] = jumlah;
    }
}

// pembanding CPU satu thread, urutan loop i-k-j agar ramah cache (Bab 9)
static void matmul_cpu(const float *A, const float *B, float *C, int n) {
    memset(C, 0, (size_t)n * n * sizeof(float));
    for (int i = 0; i < n; i++)
        for (int k = 0; k < n; k++) {
            float a = A[(size_t)i * n + k];
            for (int j = 0; j < n; j++) C[(size_t)i * n + j] += a * B[(size_t)k * n + j];
        }
}

int main(int argc, char **argv) {
    int n = argc > 1 ? atoi(argv[1]) : 1024;
    int b = argc > 2 ? atoi(argv[2]) : 16;
    bool cpu_penuh = (argc > 3 && strcmp(argv[3], "cpu") == 0) || n <= 1024;
    size_t bytes = (size_t)n * n * sizeof(float);

    float *A = (float *)malloc(bytes), *B = (float *)malloc(bytes), *C = (float *)malloc(bytes);
    for (size_t i = 0; i < (size_t)n * n; i++) {
        A[i] = (float)((i * 7) % 13) / 13.0f - 0.5f;
        B[i] = (float)((i * 11) % 17) / 17.0f - 0.5f;
    }

    float *dA, *dB, *dC;
    CEK(cudaMalloc(&dA, bytes)); CEK(cudaMalloc(&dB, bytes)); CEK(cudaMalloc(&dC, bytes));
    cudaEvent_t e0, e1;
    CEK(cudaEventCreate(&e0)); CEK(cudaEventCreate(&e1));
    float h2d_ms, kernel_ms, d2h_ms;

    CEK(cudaEventRecord(e0));
    CEK(cudaMemcpy(dA, A, bytes, cudaMemcpyHostToDevice));
    CEK(cudaMemcpy(dB, B, bytes, cudaMemcpyHostToDevice));
    CEK(cudaEventRecord(e1));
    CEK(cudaEventSynchronize(e1));
    CEK(cudaEventElapsedTime(&h2d_ms, e0, e1));

    // peluncuran
    dim3 blok(b, b);
    dim3 grid((n + b - 1) / b, (n + b - 1) / b);
    matmul<<<grid, blok>>>(dA, dB, dC, n);               // pemanasan
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        // contoh: block 64x64 = 4096 thread melebihi batas 1024 thread per block
        fprintf(stderr, "peluncuran gagal (block %dx%d = %d thread): %s\n",
                b, b, b * b, cudaGetErrorString(err));
        return 1;
    }
    CEK(cudaDeviceSynchronize());

    std::vector<float> tk;
    for (int r = 0; r < 5; r++) {
        CEK(cudaEventRecord(e0));
        matmul<<<grid, blok>>>(dA, dB, dC, n);
        CEK(cudaEventRecord(e1));
        CEK(cudaEventSynchronize(e1));
        float ms; CEK(cudaEventElapsedTime(&ms, e0, e1));
        tk.push_back(ms);
    }
    std::sort(tk.begin(), tk.end());
    kernel_ms = tk[tk.size() / 2];                         // median

    CEK(cudaEventRecord(e0));
    CEK(cudaMemcpy(C, dC, bytes, cudaMemcpyDeviceToHost));
    CEK(cudaEventRecord(e1));
    CEK(cudaEventSynchronize(e1));
    CEK(cudaEventElapsedTime(&d2h_ms, e0, e1));

    // Periksa kebenaran. Untuk n besar, CPU penuh sangat lambat (n^3), jadi cukup
    // periksa 256 elemen acak dengan perkalian titik langsung.
    double cpu_ms = -1, galat = 0, norma = 0;
    if (cpu_penuh) {
        float *ref = (float *)malloc(bytes);
        auto t0 = std::chrono::steady_clock::now();
        matmul_cpu(A, B, ref, n);
        auto t1 = std::chrono::steady_clock::now();
        cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
        for (size_t i = 0; i < (size_t)n * n; i++) {
            galat = fmax(galat, fabs((double)C[i] - ref[i]));
            norma = fmax(norma, fabs((double)ref[i]));
        }
        free(ref);
    } else {
        srand(1);
        for (int s = 0; s < 256; s++) {
            int i = rand() % n, j = rand() % n;
            double r = 0;
            for (int k = 0; k < n; k++) r += (double)A[(size_t)i * n + k] * B[(size_t)k * n + j];
            galat = fmax(galat, fabs((double)C[(size_t)i * n + j] - r));
            norma = fmax(norma, fabs(r));
        }
    }
    double galat_rel = galat / fmax(norma, 1e-30);
    double gflops = 2.0 * n * (double)n * n / (kernel_ms * 1e6);
    float total_ms = h2d_ms + kernel_ms + d2h_ms;

    fprintf(stderr, "matmul naif n=%d block %dx%d: kernel %.3f ms, %.1f GFLOPS, galat relatif %.2e (%s)\n",
            n, b, b, kernel_ms, gflops, galat_rel, galat_rel < 1e-4 ? "benar" : "PERIKSA");
    printf("n,blok,cpu_ms,h2d_ms,kernel_ms,d2h_ms,gpu_total_ms,GFLOPS,galat_relatif\n");
    printf("%d,%d,%.1f,%.3f,%.3f,%.3f,%.3f,%.1f,%.2e\n", n, b, cpu_ms, h2d_ms, kernel_ms, d2h_ms,
           total_ms, gflops, galat_rel);

    CEK(cudaFree(dA)); CEK(cudaFree(dB)); CEK(cudaFree(dC));
    free(A); free(B); free(C);
    return galat_rel < 1e-4 ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=sm_75 matmul.cu -o matmul
!for n in 256 512 1024 2048; do ./matmul $n 16 2>/dev/null | tail -1; done
!for b in 8 16 32; do ./matmul 1024 $b 2>/dev/null | tail -1; done
!./matmul 512 64

## Versi Python: CuPy dan Numba (Kode 11.9, 11.10)

Colab biasanya sudah memasang CuPy. Numba CUDA dipasang lewat paket `numba-cuda`.

In [ ]:
!pip install -q 'numba-cuda[cu12]'

In [ ]:
%%writefile vadd_cupy.py
"""Penjumlahan vektor dengan CuPy: operasi array dan RawKernel (Kode 11.9).

Pemasangan (CUDA 12.x): pip install cupy-cuda12x
Jalankan: python vadd_cupy.py
"""
import time

import cupy as cp
import numpy as np

n = 1 << 24
a = cp.ones(n, dtype=cp.float32)          # langsung dibuat di memori GPU
b = cp.full(n, 2.0, dtype=cp.float32)
c = a + b                                 # kernel elementwise bawaan CuPy

tambah = cp.RawKernel(r'''
extern "C" __global__ void tambah(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}''', "tambah")
c2 = cp.empty_like(a)
tambah(((n + 255) // 256,), (256,), (a, b, c2, cp.int32(n)))   # grid, block, argumen
print(float(c[0]), float(c2[-1]))         # salin ke host hanya saat dicetak

# Periksa semua elemen di GPU, lalu ukur waktu kernel dengan CUDA event
assert bool(cp.all(c2 == 3.0)), "hasil RawKernel salah"
mulai, selesai = cp.cuda.Event(), cp.cuda.Event()
mulai.record()
for _ in range(10):
    tambah(((n + 255) // 256,), (256,), (a, b, c2, cp.int32(n)))
selesai.record()
selesai.synchronize()
ms = cp.cuda.get_elapsed_time(mulai, selesai) / 10
print(f"RawKernel: {ms:.3f} ms, bandwidth efektif {3 * a.nbytes / ms / 1e6:.1f} GB/s")

# Bandingkan biaya transfer: salin dari NumPy (host) ke GPU dan kembali
x = np.ones(n, dtype=np.float32)
for ulang in range(2):                    # putaran pertama = pemanasan (alokasi, inisialisasi)
    t0 = time.perf_counter()
    dx = cp.asarray(x)                    # host ke device
    hasil = cp.asnumpy(dx + dx)           # kernel + device ke host
    cp.cuda.Stream.null.synchronize()
    dt = (time.perf_counter() - t0) * 1e3
print(f"salin masuk + kernel + salin keluar: {dt:.1f} ms, hasil[0] = {hasil[0]}")


In [ ]:
!python vadd_cupy.py

In [ ]:
%%writefile vadd_numba.py
"""Penjumlahan vektor dengan Numba CUDA (Kode 11.10).

Pemasangan: pip install numba-cuda[cu12]   (dukungan CUDA Numba kini di paket numba-cuda)
Jalankan: python vadd_numba.py
"""
import numpy as np
from numba import cuda


@cuda.jit
def tambah(a, b, c):
    i = cuda.grid(1)                       # = blockIdx.x * blockDim.x + threadIdx.x
    if i < c.size:
        c[i] = a[i] + b[i]


n = 1 << 24
da = cuda.to_device(np.ones(n, dtype=np.float32))       # salin ke GPU
db = cuda.to_device(np.full(n, 2.0, dtype=np.float32))
dc = cuda.device_array(n, dtype=np.float32)
tambah[(n + 255) // 256, 256](da, db, dc)               # [grid, block]
print(dc.copy_to_host()[:3])

# periksa semua elemen dan ukur waktu kernel (peluncuran pertama memuat JIT)
assert np.all(dc.copy_to_host() == 3.0), "hasil salah"
mulai, selesai = cuda.event(), cuda.event()
mulai.record()
for _ in range(10):
    tambah[(n + 255) // 256, 256](da, db, dc)
selesai.record()
selesai.synchronize()
ms = cuda.event_elapsed_time(mulai, selesai) / 10
print(f"kernel Numba: {ms:.3f} ms, bandwidth efektif {3 * 4 * n / ms / 1e6:.1f} GB/s")


In [ ]:
!python vadd_numba.py

## Pertanyaan analisis

- Pada n berapa speedup total penjumlahan vektor melewati 1? Bagaimana dengan perkalian matriks? Mengapa berbeda?
- Berapa bandwidth efektif kernel dibandingkan bandwidth memori GPU di spesifikasinya?